Dataset creation

Data integrity checks (must-do)
•	Ensure each Filename appears for:
o	one Ground_Truth transcript (or tag it as ASR_Model="Human")
o	and exactly 8 ASR rows
•	Check Group ∈ {Healthy, Dementia} and class balance


In [ ]:
import pandas as pd

df = pd.read_csv("../data/All_data.csv")

print(df.shape)
df.head()

In [ ]:
import re

def _tokenize_words(text: str):
    # Simple whitespace tokenization but normalize spaces
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []

def cap_repeated_ngrams(
    text: str,
    max_repeats: int = 3,
    min_n: int = 2,
    max_n: int = 5,
    also_cap_single_words: bool = True,
    max_word_repeats: int = 4,
):
    """
    Caps any repeated n-gram pattern (n in [min_n, max_n]) to max_repeats if it repeats consecutively.
    Example: "a boy a boy a boy a boy" with n=2 -> keep only 3 repeats.
    Also optionally caps single-word repeats (e.g., "the the the...") to max_word_repeats.
    """
    toks = _tokenize_words(text)
    if not toks:
        return ""

    out = []
    i = 0
    L = len(toks)

    while i < L:
        matched = False

        # Try longer patterns first to avoid breaking longer loops into shorter ones
        for n in range(min(max_n, L - i), min_n - 1, -1):
            pattern = toks[i:i+n]

            # Count how many times this exact pattern repeats consecutively from i
            k = 1
            while i + (k+1)*n <= L and toks[i + k*n : i + (k+1)*n] == pattern:
                k += 1

            if k > max_repeats:
                # Keep only max_repeats copies
                out.extend(pattern * max_repeats)
                i += k * n
                matched = True
                break

        if matched:
            continue

        # Optionally cap single-word repeats (helps "uh uh uh ..." etc.)
        if also_cap_single_words:
            w = toks[i]
            k = 1
            while i + k < L and toks[i + k] == w:
                k += 1
            if k > max_word_repeats:
                out.extend([w] * max_word_repeats)
                i += k
                continue

        out.append(toks[i])
        i += 1

    return " ".join(out)

In [ ]:
df.loc[df['ASR_Model'] == 'Microsoft', 'Prediction'] = (
    df.loc[df['ASR_Model'] == 'Microsoft', 'Prediction']
      .apply(lambda t: cap_repeated_ngrams(t, max_repeats=3, min_n=2, max_n=5))
)

In [ ]:
df['word_count'] = df['Prediction'].astype(str).str.split().str.len()
print(df.groupby('ASR_Model')['word_count'].describe())

In [ ]:
df['Task'] = df['Task'].astype(str).str.strip().str.upper()

task_map = {
    'COOK': 'PICT',
    'FREE': 'MONO',
    'STOR': 'MONO',
    'MONO': 'MONO',
    'PICT': 'PICT'
}

df['Task'] = df['Task'].replace(task_map)

df['Task'].value_counts()

In [ ]:
df['Group'] = df['Group'].astype(str).str.strip().str.upper()

# Map to readable labels
df['Group'] = df['Group'].replace({
    'OLD': 'Healthy',
    'DEM': 'Dementia'
})

In [ ]:
df['Group'].value_counts()

In [ ]:
transcript_counts = df.groupby("Filename")["ASR_Model"].count()

print(transcript_counts.describe())

In [ ]:
transcript_counts.value_counts()

In [ ]:
df['ASR_Model'].value_counts()

In [ ]:
asr_per_file = df.groupby("Filename")["ASR_Model"].nunique()

asr_per_file.value_counts()

In [ ]:
file_labels = df[['Filename','Group']].drop_duplicates()

file_labels['Group'].value_counts()

In [ ]:
file_labels['Group'].value_counts(normalize=True)

In [ ]:
df['text_length'] = df['Prediction'].str.split().str.len()

df.groupby('ASR_Model')['text_length'].describe()

In [ ]:
df['word_count'] = df['Prediction'].str.split().str.len()

summary = df.groupby('ASR_Model')['word_count'].describe()

print(summary)

In [ ]:
df['gt_length'] = df['Ground_Truth'].str.split().str.len()

df['gt_length'].describe()

Combined transcript length summary

In [ ]:
# Word counts
df['gt_length'] = df['Ground_Truth'].str.split().str.len()
df['asr_length'] = df['Prediction'].str.split().str.len()

# Remove duplicates for human transcripts
human_lengths = df[['Filename','gt_length']].drop_duplicates()

human_summary = human_lengths['gt_length'].describe().to_frame().T
human_summary.index = ['Human']

# ASR summaries
asr_summary = df.groupby('ASR_Model')['asr_length'].describe()

# Combine
combined_summary = pd.concat([human_summary, asr_summary])

print(combined_summary)

In [ ]:
combined_summary = combined_summary.reset_index().rename(columns={
    'index': 'Source',
    'count': 'N',
    'mean': 'Mean_Words',
    'std': 'SD',
    'min': 'Min',
    '25%': 'Q1',
    '50%': 'Median',
    '75%': 'Q3',
    'max': 'Max'
})

In [ ]:
combined_summary

In [ ]:
combined_summary.to_csv("transcript_length_summary.csv", index=False)

In [ ]:
df.isnull().sum()

In [ ]:
human_counts = df.groupby('Filename')['Ground_Truth'].nunique()

human_counts.value_counts()

In [ ]:
# human transcripts (now includes Task)
human_df = df[['Filename', 'Group', 'Task', 'Ground_Truth']].drop_duplicates()
human_df = human_df.rename(columns={'Ground_Truth': 'Transcript'})
human_df['Source'] = 'Human'

# ASR transcripts (now includes Task)
asr_df = df[['Filename', 'Group', 'Task', 'Prediction', 'ASR_Model']].copy()
asr_df = asr_df.rename(columns={'Prediction': 'Transcript', 'ASR_Model': 'Source'})

# combine (reset index to keep it tidy)
final_df = pd.concat([human_df, asr_df], ignore_index=True)

print(final_df.shape)
final_df.head()

Experiment A — “Clinical impact”: how much ASR changes diagnosis performance

A1. Train on Human, test on Human vs ASR (most persuasive)
Train a transcript-based dementia classifier using Ground_Truth text only.
Then evaluate on:
•	Human transcripts (upper bound)
•	Each ASR transcript (Prediction) for the same test files
Outputs per ASR:
•	AUC, macro-F1, balanced accuracy
•	ΔAUC = AUC(ASR) − AUC(Human)
•	ΔF1 = F1(ASR) − F1(Human)
Why this nails the editor feedback
It directly measures: “If the hospital uses ASR instead of human transcription, how much does dementia detection degrade?”


Experiment 1
Train on Human transcripts → Test on Human + ASRs
Goal

Measure how dementia classification performance changes when transcripts come from ASR systems instead of humans.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score
)

In [ ]:
# Normalize + map your dataset's group labels

final_df['label'] = final_df['Group'].map({'Healthy': 0, 'Dementia': 1})

# Sanity checks
print(final_df['Group'].value_counts())
print("Missing labels:", final_df['label'].isna().sum())

# Rebuild human_df (one row per filename)
human_df = final_df[final_df['Source'] == "Human"].copy()
human_df = human_df.drop_duplicates(subset=['Filename'], keep='first')

X = human_df['Transcript']
y = human_df['label']
filenames = human_df['Filename']

print("Human label counts:\n", human_df['Group'].value_counts())

In [ ]:
# convert labels to numeric
final_df['label'] = final_df['Group'].map({
    'Healthy':0,
    'Dementia':1
})

In [ ]:
human_df = final_df[final_df['Source'] == "Human"].copy()

In [ ]:
human_df['Group'].value_counts()

In [ ]:
X = human_df['Transcript']
y = human_df['label']
filenames = human_df['Filename']

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
asr_sources = final_df['Source'].unique()

results = {src:[] for src in asr_sources}

In [ ]:
for train_idx, test_idx in skf.split(X, y):

    train_files = filenames.iloc[train_idx]
    test_files = filenames.iloc[test_idx]

    train_text = human_df.iloc[train_idx]['Transcript']
    train_labels = y.iloc[train_idx]

    # vectorizer
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    )

    X_train = vectorizer.fit_transform(train_text)

    model = LogisticRegression(max_iter=2000)

    model.fit(X_train, train_labels)

    # evaluate on each transcript source
    for src in asr_sources:

        test_subset = final_df[
            (final_df['Filename'].isin(test_files)) &
            (final_df['Source'] == src)
        ]

        X_test = vectorizer.transform(test_subset['Transcript'])
        y_test = test_subset['label']

        probs = model.predict_proba(X_test)[:,1]
        preds = (probs > 0.5).astype(int)

        auc = roc_auc_score(y_test, probs)
        f1 = f1_score(y_test, preds)
        bal_acc = balanced_accuracy_score(y_test, preds)

        results[src].append({
            'AUC':auc,
            'F1':f1,
            'BalancedAcc':bal_acc
        })

In [ ]:
summary = []

for src,vals in results.items():

    auc = np.mean([v['AUC'] for v in vals])
    f1 = np.mean([v['F1'] for v in vals])
    bal = np.mean([v['BalancedAcc'] for v in vals])

    summary.append({
        'Source':src,
        'AUC':auc,
        'F1':f1,
        'BalancedAcc':bal
    })

summary_df = pd.DataFrame(summary).sort_values("AUC",ascending=False)

print(summary_df)

In [ ]:
human_auc = summary_df.loc[summary_df['Source']=='Human','AUC'].values[0]

summary_df['Delta_AUC_vs_Human'] = summary_df['AUC'] - human_auc

summary_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

sns.barplot(
    data=summary_df,
    x='Source',
    y='AUC'
)

plt.axhline(human_auc, linestyle='--', color='red', label='Human baseline')

plt.xticks(rotation=45)
plt.legend()
plt.title("Dementia classification performance across transcript sources")

plt.show()

Experiment 1 results for stats

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Ensure 1 human row per file (file-level label + task)
human_df = final_df[final_df['Source'] == 'Human'].copy()
human_df = human_df.drop_duplicates(subset=['Filename'], keep='first')

X = human_df['Transcript'].astype(str)
y = human_df['label'].astype(int)
filenames = human_df['Filename'].astype(str)

# File-level metadata (rename Task to avoid merge collisions)
meta = human_df[['Filename', 'Task', 'Group', 'label']].copy()
meta = meta.rename(columns={'Task': 'Task_meta', 'label': 'y_true'})

sources = sorted(final_df['Source'].unique())
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
fold_id = 0

for train_idx, test_idx in skf.split(X, y):
    fold_id += 1

    test_files = filenames.iloc[test_idx]
    train_text = X.iloc[train_idx]
    train_labels = y.iloc[train_idx]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9
    )
    X_train = vectorizer.fit_transform(train_text)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, train_labels)

    for src in sources:
        test_subset = final_df[
            (final_df['Filename'].isin(test_files)) &
            (final_df['Source'] == src)
        ].copy()

        if test_subset.empty:
            continue

        # Only keep what we need from test_subset to avoid Task collision
        test_subset = test_subset[['Filename', 'Transcript', 'Source']].copy()

        # Attach file-level metadata (Task_meta, Group, y_true)
        test_subset = test_subset.merge(meta, on='Filename', how='left')
        test_subset = test_subset.dropna(subset=['y_true'])

        X_test = vectorizer.transform(test_subset['Transcript'].astype(str))
        y_prob = clf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        rows.append(pd.DataFrame({
            'fold': fold_id,
            'Filename': test_subset['Filename'].astype(str).values,
            'Task': test_subset['Task_meta'].astype(str).values,
            'Group': test_subset['Group'].astype(str).values,
            'y_true': test_subset['y_true'].astype(int).values,
            'Source': src,
            'y_prob': y_prob,
            'y_pred': y_pred
        }))

pred_long = pd.concat(rows, ignore_index=True)

print(pred_long.shape)
pred_long.head()

In [ ]:
pred_long.to_csv("exp1_filewise_predictions_long.csv", index=False)
print("Saved: exp1_filewise_predictions_long.csv")

In [ ]:
# Expect each (fold, Filename) to have all sources (e.g., 9)
print(pred_long.groupby(['fold','Filename'])['Source'].nunique().value_counts().head(10))

# Check tasks are stable per file
print("Files with >1 task label:",
      (pred_long.groupby('Filename')['Task'].nunique() > 1).sum())

Experiment 2 — Clinical risk / error-profile analysis
Goal

Using the same setup as Experiment 1 (train on Human transcripts only), quantify how each ASR changes:

Sensitivity / Recall for Dementia (TPR)

Specificity for Healthy (TNR)

False Negative Rate for Dementia (FNR = 1 − TPR) ← clinically critical

False Positive Rate for Healthy (FPR = 1 − TNR)

We’ll compute these file-wise and fold-wise, then aggregate across folds with confidence intervals (optional but recommended).

You already exported exp1_filewise_predictions_long.csv, which is perfect — we can compute Experiment 2 directly from that file (no need to rerun models).


In [ ]:
import pandas as pd
import numpy as np

pred_long = pd.read_csv("exp1_filewise_predictions_long.csv")
pred_long.head()

In [ ]:
def rates_from_df(g: pd.DataFrame):
    # dementia is positive class = 1
    y_true = g['y_true'].astype(int).values
    y_pred = g['y_pred'].astype(int).values

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))

    # avoid divide-by-zero
    TPR = TP / (TP + FN) if (TP + FN) > 0 else np.nan  # sensitivity
    TNR = TN / (TN + FP) if (TN + FP) > 0 else np.nan  # specificity
    FNR = 1 - TPR if not np.isnan(TPR) else np.nan
    FPR = 1 - TNR if not np.isnan(TNR) else np.nan

    return pd.Series({
        'TP': TP, 'FN': FN, 'TN': TN, 'FP': FP,
        'TPR_sens': TPR,
        'TNR_spec': TNR,
        'FNR': FNR,
        'FPR': FPR
    })

fold_src_rates = pred_long.groupby(['fold','Source']).apply(rates_from_df).reset_index()

fold_src_rates.head()

In [ ]:
summary = (fold_src_rates
           .groupby('Source')[['TPR_sens','TNR_spec','FNR','FPR']]
           .agg(['mean','std'])
           .reset_index())

summary

In [ ]:
summary_clean = fold_src_rates.groupby('Source')[['TPR_sens','TNR_spec','FNR','FPR']].mean().reset_index()
summary_clean

In [ ]:
human_row = summary_clean[summary_clean['Source']=='Human'].iloc[0]

for col in ['TPR_sens','TNR_spec','FNR','FPR']:
    summary_clean[f'Delta_{col}_vs_Human'] = summary_clean[col] - human_row[col]

summary_clean.sort_values('Delta_FNR_vs_Human', ascending=False)

In [ ]:
fold_src_rates.to_csv("exp2_fold_source_error_rates.csv", index=False)
summary_clean.to_csv("exp2_error_rates_summary_vs_human.csv", index=False)

print("Saved: exp2_fold_source_error_rates.csv")
print("Saved: exp2_error_rates_summary_vs_human.csv")

Experiment C — “Biomarker validity”: do ASRs distort dementia-linked text markers?
Compute interpretable features on both Ground_Truth and Prediction.
Feature set (fast, defensible)
Lexical/fluency markers that are standard in dementia-from-language work:
•	total words, mean sentence length
•	type-token ratio / MATTR
•	content word ratio (nouns+verbs+adjectives / all)
•	pronoun ratio (pronouns / all)
•	repetition rate (exact repeated unigrams/bigrams)
•	filler counts (uh/um) if present
•	pause tokens if your transcripts contain them (many don’t)
Three biomarker analyses reviewers like
1.	Feature distortion: For each feature and ASR, report correlation with Human across files (Spearman).
2.	Effect-size attenuation: Cohen’s d (Dementia vs Healthy) computed using Human features vs ASR features; show shrinkage.
3.	Predictive validity drop: Train a simple logistic regression on Human features; test on ASR features.
This shows “digital biomarker validity” damage, not just WER.


In [ ]:
import pandas as pd
import numpy as np
import re

df = final_df.copy()

df.head()

In [ ]:
def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"\b[a-z']+\b", text)
    return tokens

In [ ]:
pronouns = {
"i","me","my","mine","you","your","yours","he","him","his",
"she","her","hers","it","its","we","us","our","ours",
"they","them","their","theirs"
}

function_words = {
"the","a","an","in","on","at","for","to","of","and","but","or",
"is","are","was","were","be","been","being"
}

In [ ]:
def extract_features(text):

    tokens = tokenize(text)

    if len(tokens) == 0:
        return pd.Series({
            "n_words":0,
            "ttr":0,
            "pronoun_ratio":0,
            "content_ratio":0,
            "mean_word_len":0
        })

    unique = len(set(tokens))

    pronoun_count = sum(t in pronouns for t in tokens)
    function_count = sum(t in function_words for t in tokens)

    content_count = len(tokens) - function_count

    return pd.Series({
        "n_words": len(tokens),
        "ttr": unique / len(tokens),
        "pronoun_ratio": pronoun_count / len(tokens),
        "content_ratio": content_count / len(tokens),
        "mean_word_len": np.mean([len(t) for t in tokens])
    })

In [ ]:
features = df["Transcript"].apply(extract_features)

bio_df = pd.concat([df[['Filename','Group','Source']], features], axis=1)

bio_df.head()

In [ ]:
bio_df.to_csv("exp3_filewise_predictions_long.csv", index=False)


In [ ]:
def effect_size(g):

    healthy = g[g["Group"]=="Healthy"]
    dementia = g[g["Group"]=="Dementia"]

    results = {}

    for col in ["ttr","pronoun_ratio","content_ratio","mean_word_len"]:

        h = healthy[col].mean()
        d = dementia[col].mean()

        results[col+"_effect"] = d - h

    return pd.Series(results)

effects = bio_df.groupby("Source").apply(effect_size).reset_index()

effects

In [ ]:
human_row = effects[effects.Source=="Human"].iloc[0]

for col in effects.columns[1:]:

    effects[f"delta_{col}_vs_human"] = effects[col] - human_row[col]

effects

In [ ]:
effects.to_csv("exp3_biomarker_distortion.csv", index=False)

Plots for the experiments

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.2)

# consistent color palette
palette = {
    "Human":"black",
    "Google":"#1f77b4",
    "CrisperWhisper":"#ff7f0e",
    "AWS":"#2ca02c",
    "AssemblyAI":"#d62728",
    "Whisper":"#9467bd",
    "Nemo":"#8c564b",
    "Microsoft":"#e377c2",
    "Meta":"#7f7f7f"
}

fig, axes = plt.subplots(1,3, figsize=(18,5))

# ---------------------
# Panel A — AUC
# ---------------------

auc_data = pd.DataFrame({
    "Source":["Human","Google","CrisperWhisper","AWS","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
    "AUC":[0.8917,0.8777,0.8747,0.8715,0.8665,0.8664,0.8486,0.8396,0.8391]
})

sns.barplot(
    data=auc_data,
    x="Source",
    y="AUC",
    palette=palette,
    ax=axes[0]
)

axes[0].set_title("A. Dementia Classification Performance")
axes[0].set_ylabel("AUC")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=45)


# ---------------------
# Panel B — Error profile
# ---------------------

error_data = pd.DataFrame({
    "Source":["Human","Google","AWS","CrisperWhisper","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
    "FNR":[0.0367,0.0259,0.0575,0.0627,0.1040,0.1094,0.0835,0.1042,0.1298],
    "FPR":[0.5122,0.5577,0.5039,0.5122,0.4233,0.4411,0.4862,0.4949,0.4502]
})

sns.scatterplot(
    data=error_data,
    x="FPR",
    y="FNR",
    hue="Source",
    palette=palette,
    s=120,
    ax=axes[1],
    legend=False
)

axes[1].set_title("B. Diagnostic Error Profile")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("False Negative Rate")


# ---------------------
# Panel C — Biomarker distortion
# ---------------------

bio_data = pd.DataFrame({
    "Feature":["TTR","Pronoun","Content","WordLen"],
    "Google":[0.0097,0.0014,-0.0113,-0.0207],
    "AWS":[0.0184,0.0068,-0.0104,-0.0209],
    "CrisperWhisper":[-0.0046,0.0018,-0.0099,-0.0369],
    "Meta":[0.0749,-0.0092,-0.0064,0.0204]
})

bio_data = bio_data.melt(id_vars="Feature", var_name="Source", value_name="Delta")

sns.barplot(
    data=bio_data,
    x="Feature",
    y="Delta",
    hue="Source",
    palette=palette,
    ax=axes[2]
)

axes[2].set_title("C. Biomarker Distortion")
axes[2].set_ylabel("Effect Size Difference vs Human")


# ---------------------
# Shared legend
# ---------------------

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=5,
    frameon=False
)

for ax in axes:
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

plt.tight_layout(rect=[0,0,1,0.92])
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.2)

palette = {
    "Human":"black",
    "Google":"#1f77b4",
    "CrisperWhisper":"#ff7f0e",
    "AWS":"#2ca02c",
    "AssemblyAI":"#d62728",
    "Whisper":"#9467bd",
    "Nemo":"#8c564b",
    "Microsoft":"#e377c2",
    "Meta":"#7f7f7f"
}

fig, axes = plt.subplots(1,3, figsize=(20,6))

# --------------------
# Panel A: AUC
# --------------------

auc_data = pd.DataFrame({
"Source":["Human","Google","CrisperWhisper","AWS","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
"AUC":[0.8917,0.8777,0.8747,0.8715,0.8665,0.8664,0.8486,0.8396,0.8391]
})

sns.barplot(data=auc_data, x="Source", y="AUC", palette=palette, ax=axes[0])
axes[0].set_title("A. Dementia Classification Performance")
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_xlabel("")
axes[0].set_ylabel("AUC")

# --------------------
# Panel B: Error profile
# --------------------

error_data = pd.DataFrame({
"Source":["Human","Google","AWS","CrisperWhisper","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
"FNR":[0.0367,0.0259,0.0575,0.0627,0.1040,0.1094,0.0835,0.1042,0.1298],
"FPR":[0.5122,0.5577,0.5039,0.5122,0.4233,0.4411,0.4862,0.4949,0.4502]
})

sns.scatterplot(
    data=error_data,
    x="FPR",
    y="FNR",
    hue="Source",
    palette=palette,
    s=120,
    ax=axes[1]
)

axes[1].set_title("B. Diagnostic Error Profile")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("False Negative Rate")

# --------------------
# Panel C: Biomarker distortion
# --------------------

bio_data = pd.DataFrame({
"Source":["AWS","AssemblyAI","CrisperWhisper","Google","Meta","Microsoft","Nemo","Whisper"],
"TTR":[0.018456,0.041549,-0.004645,0.009751,0.074988,0.030333,0.021767,0.025299],
"Pronoun":[0.006857,0.008502,0.001834,0.001452,-0.009248,0.005934,0.002852,0.006370],
"Content":[-0.010395,-0.012841,-0.009932,-0.011346,-0.006403,-0.027252,-0.010485,-0.011613]
})

bio_data = bio_data.melt(id_vars="Source", var_name="Feature", value_name="Delta")

sns.barplot(
    data=bio_data,
    x="Feature",
    y="Delta",
    hue="Source",
    palette=palette,
    ax=axes[2]
)

axes[2].set_title("C. Biomarker Distortion")
axes[2].set_ylabel("Effect Size Difference vs Human")

plt.tight_layout()
plt.show()

In [ ]:



# --------------------
# SINGLE LEGEND (RIGHT)
# --------------------

handles, labels = axes[1].get_legend_handles_labels()

for ax in axes:
    legend = ax.get_legend()
    if legend:
        legend.remove()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.02,0.5),
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.2)

palette = {
    "Human":"black",
    "Google":"#1f77b4",
    "CrisperWhisper":"#ff7f0e",
    "AWS":"#2ca02c",
    "AssemblyAI":"#d62728",
    "Whisper":"#9467bd",
    "Nemo":"#8c564b",
    "Microsoft":"#e377c2",
    "Meta":"#7f7f7f"
}

fig, axes = plt.subplots(1,3, figsize=(20,6))

# --------------------
# Panel A
# --------------------

auc_data = pd.DataFrame({
"Source":["Human","Google","CrisperWhisper","AWS","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
"AUC":[0.8917,0.8777,0.8747,0.8715,0.8665,0.8664,0.8486,0.8396,0.8391]
})

sns.barplot(data=auc_data, x="Source", y="AUC", palette=palette, ax=axes[0])

axes[0].set_ylim(0.6,1)
axes[0].set_title("A. Dementia Classification Performance")
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_xlabel("")
axes[0].set_ylabel("AUC")

# --------------------
# Panel B
# --------------------

error_data = pd.DataFrame({
"Source":["Human","Google","AWS","CrisperWhisper","AssemblyAI","Whisper","Nemo","Microsoft","Meta"],
"FNR":[0.0367,0.0259,0.0575,0.0627,0.1040,0.1094,0.0835,0.1042,0.1298],
"FPR":[0.5122,0.5577,0.5039,0.5122,0.4233,0.4411,0.4862,0.4949,0.4502]
})

sns.scatterplot(
    data=error_data,
    x="FPR",
    y="FNR",
    hue="Source",
    palette=palette,
    s=120,
    ax=axes[1]
)

axes[1].set_title("B. Diagnostic Error Profile")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("False Negative Rate")

# --------------------
# Panel C
# --------------------

bio_data = pd.DataFrame({
"Source":["AWS","AssemblyAI","CrisperWhisper","Google","Meta","Microsoft","Nemo","Whisper"],
"TTR":[0.018456,0.041549,-0.004645,0.009751,0.074988,0.030333,0.021767,0.025299],
"Pronoun":[0.006857,0.008502,0.001834,0.001452,-0.009248,0.005934,0.002852,0.006370],
"Content":[-0.010395,-0.012841,-0.009932,-0.011346,-0.006403,-0.027252,-0.010485,-0.011613]
})

bio_data = bio_data.melt(id_vars="Source", var_name="Feature", value_name="Delta")

sns.barplot(
    data=bio_data,
    x="Feature",
    y="Delta",
    hue="Source",
    palette=palette,
    ax=axes[2]
)

axes[2].set_title("C. Biomarker Distortion")
axes[2].set_ylabel("Effect Size Difference vs Human") 
# --------------------
# SINGLE LEGEND (BOTTOM)
# --------------------

handles, labels = axes[1].get_legend_handles_labels()

for ax in axes:
    legend = ax.get_legend()
    if legend:
        legend.remove()

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5,-0.05),
    ncol=5,
    frameon=False
)

plt.tight_layout(rect=[0,0.05,1,1])
plt.show()